# Phase 3: Advanced CoT Techniques - Executable
## Self-Consistency, Tree-of-Thoughts, and Beyond

Build and test advanced reasoning techniques.

In [ ]:
import os
import re
import json
from collections import Counter
from typing import List, Dict
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

print("✅ Imports successful")

In [ ]:
# Load local Mistral-7B model (same as Phase 1 & 2)
from transformers import pipeline
import torch

print("Loading Mistral-7B model (local inference)...")

try:
    device = 0 if torch.cuda.is_available() else -1
    device_str = "GPU" if device == 0 else "CPU"
    print(f"Using: {device_str}")
    
    generator = pipeline(
        "text-generation",
        model="mistralai/Mistral-7B-Instruct-v0.1",
        torch_dtype=torch.float16 if device == 0 else torch.float32,
        device_map="auto" if device == 0 else None,
        device=device
    )
    
    print("✅ Mistral-7B model loaded!")
    
    def call_model(prompt: str, temperature: float = 0.0, max_tokens: int = 300) -> str:
        """Generate text using local Mistral-7B model"""
        try:
            use_sampling = temperature > 0.0
            actual_temp = max(temperature, 0.1) if not use_sampling else temperature
            
            response = generator(
                prompt,
                max_new_tokens=max_tokens,
                temperature=actual_temp,
                num_return_sequences=1,
                do_sample=use_sampling
            )
            return response[0]["generated_text"]
        except Exception as e:
            raise e
    
    print("✅ Model configured")
    
except Exception as e:
    print(f"Error: {e}")
    print("Fallback: Using GPT-2")
    generator = pipeline("text-generation", model="gpt2", device=-1)
    
    def call_model(prompt: str, temperature: float = 0.0, max_tokens: int = 300) -> str:
        try:
            use_sampling = temperature > 0.0
            actual_temp = max(temperature, 0.1) if not use_sampling else temperature
            
            response = generator(
                prompt,
                max_new_tokens=max_tokens,
                temperature=actual_temp,
                num_return_sequences=1,
                do_sample=use_sampling
            )
            return response[0]["generated_text"]
        except Exception as e:
            raise e

## Overview: Advanced CoT Techniques

In Phase 2, you learned basic **Chain-of-Thought Prompting** - showing reasoning steps to improve accuracy. Phase 3 takes this further with three advanced techniques that push accuracy even higher.

### The Three Techniques

**1. Self-Consistency Sampling** - Generate multiple reasoning paths and vote
- Problem: "A single reasoning path might be wrong"
- Solution: "Generate 5-10 different reasoning chains, vote on answer"
- Best for: High-stakes decisions, complex problems
- Cost: 5-10x slower

**2. Least-to-Most Prompting** - Break hard problems into easier sub-problems
- Problem: "Models struggle with multi-step problems"
- Solution: "Decompose into simple steps, solve progressively"
- Best for: Multi-step, sequential problems
- Cost: 2-3x slower, more setup

**3. Confidence Scoring** - Ask model how confident it is in its answer
- Problem: "We don't know if the answer is trustworthy"
- Solution: "Ask model to rate its certainty (1-10)"
- Best for: Filtering outputs, quality assurance
- Cost: 1 extra inference call

### Quick Comparison

| Technique | Problem It Solves | Accuracy Gain | Speed Cost | Best For |
|-----------|------------------|---------------|-----------|----------|
| **Self-Consistency** | Single path might be wrong | +5-15% | 5-10x slower | Critical accuracy |
| **Least-to-Most** | Multi-step reasoning fails | +10-20% | 2-3x slower | Complex decomposable problems |
| **Confidence Scoring** | Don't know if answer is right | N/A (filtering) | 1x slower | Filtering, QA |

### When to Combine Techniques

```
Simple problem:
  → Use basic CoT (Phase 2 is enough)

Medium complexity:
  → Use Confidence Scoring (know what to trust)

High complexity:
  → Use Least-to-Most (breaks it down)

High stakes + critical accuracy:
  → Combine Self-Consistency + Confidence Scoring

Unknown complexity + safety critical:
  → Use Least-to-Most + Confidence Scoring
```

### Key Insight

Each technique addresses a **different failure mode**:
- Self-Consistency: "Different reasoning might work better"
- Least-to-Most: "Need to break problem into pieces"
- Confidence: "Need to know what to trust"

Let's explore each one in detail.

---

## Self-Consistency Sampling

### What is Self-Consistency Sampling?

**Self-Consistency Sampling** is an advanced technique that improves accuracy by:
1. Generating **multiple reasoning chains** (5-10 different attempts)
2. **Extracting answers** from each reasoning path
3. **Voting** on the most common answer

**Key Insight:** Instead of relying on a single reasoning path, we generate diverse reasoning approaches and let them "vote" on the answer. The answer that appears most frequently is likely the correct one.

### How It Works

```
Problem: "Sarah has 10 apples and gives 3 to John, how many left?"

Attempt 1 (temperature=0.7):
  → "She starts with 10. Gives 3 away. 10 - 3 = 7 apples"
  → Answer: 7

Attempt 2 (temperature=0.7):
  → "10 apples initially. 3 removed. 10 minus 3 equals 7."
  → Answer: 7

Attempt 3 (temperature=0.7):
  → "Beginning: 10 apples. Subtract the 3 given: 7 remain."
  → Answer: 7

VOTING: 7 appears 3/3 times → Confidence: 100%
FINAL ANSWER: 7 ✓
```

### Why It Works

- **Different Paths, Same Answer:** If multiple reasoning paths reach the same conclusion, confidence is high
- **Catches Mistakes:** If one reasoning path is wrong but others are correct, voting identifies the right answer
- **Emergent Agreement:** Large models show consistent reasoning on correct problems
- **Diversity at Reasoning Level:** High temperature (0.7) creates diverse intermediate steps, not just random outputs

### When to Use Self-Consistency Sampling

✅ **Use when:**
- Accuracy is critical (e.g., medical diagnosis, financial calculations)
- You have time for multiple inference calls (5-10x slower)
- Problem complexity is high (multiple reasoning paths likely)
- You need confidence metrics

❌ **Don't use when:**
- Speed is required (real-time systems)
- Budget is limited (multiple API calls = higher cost)
- Problem is simple (standard CoT works fine)

### Trade-offs

| Aspect | Standard CoT | Self-Consistency |
|--------|-------------|-----------------|
| Accuracy | Good | Better (+5-15%) |
| Speed | Fast | 5-10x slower |
| Cost | Low | 5-10x higher |
| Complexity | Simple | Moderate |
| Confidence | Yes | High quality |

---

In [ ]:
class SelfConsistencySolver:
    """Generate multiple reasoning chains and vote on answer"""
    
    def __init__(self, model_fn):
        self.model = model_fn
    
    def extract_answer(self, response: str) -> str:
        """Extract numerical answer"""
        numbers = re.findall(r'\d+', response)
        return numbers[-1] if numbers else ""
    
    def solve_with_self_consistency(self, problem: str, n_samples: int = 5, temperature: float = 0.7) -> Dict:
        """Generate multiple solutions and vote"""
        
        prompt = f"Let me work through this step by step.\n\nQ: {problem}\nA:"
        answers = []
        reasoning_chains = []
        
        print(f"Generating {n_samples} reasoning chains... ", end="", flush=True)
        
        for i in range(n_samples):
            try:
                response = self.model(prompt, temperature=temperature)
                answer = self.extract_answer(response)
                answers.append(answer)
                reasoning_chains.append(response)
            except Exception as e:
                print(f"Error: {e}")
        
        # Vote on most common answer
        if answers:
            answer_counts = Counter(answers)
            final_answer = answer_counts.most_common(1)[0][0]
            confidence = answer_counts.most_common(1)[0][1] / len(answers)
        else:
            final_answer = ""
            confidence = 0
        
        print("✓")
        
        return {
            'problem': problem,
            'final_answer': final_answer,
            'all_answers': answers,
            'reasoning_chains': reasoning_chains,
            'confidence': confidence,
            'n_samples': n_samples
        }

# Test self-consistency
solver = SelfConsistencySolver(call_model)

test_problems = [
    "If Sarah has 10 apples and gives 3 to John, how many does she have?",
    "A store sells 5 books on Monday and 8 books on Tuesday. How many books sold in total?"
]

print("\n" + "="*80)
print("SELF-CONSISTENCY SAMPLING TEST")
print("="*80)

sc_results = []
for problem in test_problems[:1]:  # Test 1 problem
    result = solver.solve_with_self_consistency(problem, n_samples=3, temperature=0.7)
    sc_results.append(result)
    
    print(f"\n📝 Problem: {problem}")
    print(f"✅ Final Answer (by voting): {result['final_answer']}")
    print(f"📊 Confidence: {result['confidence']*100:.1f}%")
    print(f"🔄 All answers: {result['all_answers']}")

### What is Least-to-Most Prompting?

**Least-to-Most Prompting** solves complex problems by:
1. **Breaking down** complex problems into simpler sub-problems
2. **Starting small** - solve the easiest parts first
3. **Building up** - solve progressively harder parts using previous solutions

**Key Insight:** Humans often solve hard problems by solving easy ones first. Models do better when they follow this pattern: easy → medium → hard.

### How It Works

```
Complex Problem: "A store has 100 items. They sell 25 in morning, 15 in 
afternoon. They receive 30 new items. Then they sell 10 more. How many now?"

STEP 1: DECOMPOSE (Break into sub-problems)
  Sub-problem 1: Items sold in morning = 25
  Sub-problem 2: Items sold in afternoon = 15
  Sub-problem 3: Total sold in day = 25 + 15 = 40
  Sub-problem 4: Items after selling = 100 - 40 = 60
  Sub-problem 5: Items received = 30
  Sub-problem 6: Items after receiving = 60 + 30 = 90
  Sub-problem 7: Items sold later = 10
  Sub-problem 8: Final total = 90 - 10 = 80

STEP 2: SOLVE PROGRESSIVELY
  Model solves #1 (easiest): 25 items
  Model uses result to solve #2 using #1: 25 + 15 = 40
  Model uses result to solve #3 using #2: 100 - 40 = 60
  ... and so on ...
  Final: 80 items
```

### Why It Works

- **Reduces Complexity:** Each step is simpler than the original problem
- **Leverages Previous Solutions:** Each step builds on previous answers
- **Matches Human Problem-Solving:** We naturally decompose hard problems
- **Fewer Errors:** Simple steps have fewer places to go wrong
- **Implicit Reasoning:** Model shows its work naturally through decomposition

### When to Use Least-to-Most Prompting

✅ **Use when:**
- Problem has clear sub-steps (multi-stage problems)
- Problems involve multiple operations in sequence
- Want to improve reasoning on complex tasks
- Need to understand the problem structure
- Model tends to "jump ahead" and make mistakes

❌ **Don't use when:**
- Problem is simple and single-step
- Sub-problems aren't clear or natural
- Decomposition would confuse the model
- Speed is critical (adds extra inference steps)

### Variants

**Variant 1: Explicit Decomposition**
- You provide the decomposition, model solves
- Use when you know the structure

**Variant 2: Implicit Decomposition**
- Model decomposes and solves automatically
- More flexible but may decompose wrong

**Variant 3: Prompted Decomposition**
- You guide decomposition with hints
- Balance between explicit and implicit

### Trade-offs

| Aspect | Standard CoT | Least-to-Most |
|--------|------------|----------------|
| Simple problems | Good | Overkill |
| Complex problems | Okay | Better |
| Transparency | Medium | High |
| Manual effort | Low | Medium |
| Inference calls | 1 | 2-3 |

---

## Least-to-Most Prompting

### What is Confidence Scoring?

**Confidence Scoring** asks the model to evaluate its own certainty by:
1. **Generating reasoning** for the problem
2. **Asking for confidence rating** (1-10 scale)
3. **Analyzing confidence** relative to correctness

**Key Insight:** Large language models can assess their own uncertainty. High confidence often correlates with correct answers; low confidence flags uncertain areas.

### How It Works

```
Problem: "A ball costs $5. Tax is 10%. What's the total?"

STEP 1: MODEL REASONS
  "The ball costs $5.
   Tax is 10% of $5 = $0.50
   Total = $5 + $0.50 = $5.50"

STEP 2: ASK FOR CONFIDENCE
  "Based on this reasoning, how confident are you (1-10)?"
  
STEP 3: MODEL RESPONDS WITH CONFIDENCE
  "I am 9/10 confident because:
   - The tax calculation is straightforward
   - Arithmetic is correct
   - This is a simple, well-defined problem"

INTERPRETATION:
  High confidence (9/10) + correct answer → Model is reliable
  Low confidence (3/10) + correct answer → Maybe lucky
  High confidence (9/10) + wrong answer → Model is overconfident
  Low confidence (3/10) + wrong answer → Model knows it's uncertain
```

### Why It Works

- **Calibrated Models:** Well-trained models know when they're uncertain
- **Early Warning:** Low confidence can flag answers needing review
- **Trust Assessment:** Helps determine if we can rely on the answer
- **Error Detection:** Overconfident answers may indicate hallucination
- **Quality Filtering:** Use confidence to automatically filter outputs

### When to Use Confidence Scoring

✅ **Use when:**
- You need to know how much to trust an answer
- Filtering outputs (keep only high-confidence answers)
- Flagging answers for human review
- Understanding model behavior
- Building reliability metrics

❌ **Don't use when:**
- Speed is critical (adds inference overhead)
- Working with very small models (may not be calibrated)
- Simple, deterministic problems (confidence always high)
- You don't need reliability information

### Confidence Levels & Interpretation

| Confidence | Interpretation | Action |
|-----------|-----------------|--------|
| 8-10 | Very confident | Trust the answer |
| 6-7 | Moderately confident | Review for accuracy |
| 4-5 | Uncertain | Needs verification |
| 1-3 | Very uncertain | Reject or verify externally |

### Variants

**Variant 1: Binary Confidence**
```
"Are you confident in this answer? Yes/No"
```
- Simplest form
- Works with smaller models

**Variant 2: Uncertainty Zones**
```
"Which zone describes your uncertainty?
 A) Very certain (no doubts)
 B) Mostly certain (minor concerns)
 C) Somewhat uncertain (some doubts)
 D) Very uncertain (major concerns)"
```
- More nuanced than binary
- Better for understanding reasoning

**Variant 3: Confidence with Reasoning**
```
"Rate confidence (1-10) and explain why:
 - What was clear/easy?
 - What was ambiguous/hard?
 - Any assumptions made?"
```
- Provides reasoning for confidence
- Helps identify weakness areas

### Trade-offs

| Aspect | Standard CoT | With Confidence |
|--------|------------|------------------|
| Answer quality | Good | Same |
| Trust level | Unknown | Known |
| Inference cost | Low | +1 extra call |
| Calibration | Not tracked | Tracked |
| Actionability | Limited | Better filtering |

---

## Decision Guide: Which Technique to Use?

After running all three techniques above, let's create a decision framework for your own problems.

### Step 1: Characterize Your Problem

Ask yourself these questions:

```
A) Problem Complexity
   Simple (1-2 steps)  → Use basic CoT
   Medium (3-5 steps)  → Consider Least-to-Most
   Complex (6+ steps)  → Use Least-to-Most

B) Criticality
   Low stakes      → Use basic CoT
   Medium stakes   → Add Confidence Scoring
   High stakes     → Use Self-Consistency + Confidence

C) Speed Requirements
   Real-time       → Can't use Self-Consistency
   < 10 seconds    → Avoid Self-Consistency
   Batch job       → Self-Consistency is okay

D) Available Budget
   Low cost        → Confidence Scoring only
   Medium cost     → Least-to-Most
   High cost       → Self-Consistency
```

### Step 2: Decision Tree

```
START: You have a reasoning task

Is the problem COMPLEX with multiple steps?
├─ YES → Use LEAST-TO-MOST PROMPTING
│        Then add Confidence Scoring
│
└─ NO → Is accuracy CRITICAL (high stakes)?
        ├─ YES → Use SELF-CONSISTENCY SAMPLING (5-10 attempts)
        │        Add Confidence Scoring for verification
        │
        └─ NO → Do you need to know if answer is trustworthy?
                ├─ YES → Use CONFIDENCE SCORING with basic CoT
                │
                └─ NO → Use BASIC CoT (Phase 2 is enough)
```

### Step 3: Practical Implementation

**For Simple + Low Stakes:**
```python
# Phase 2 approach is sufficient
response = call_model(cot_prompt, temperature=0.0)
answer = extract_answer(response)
```

**For Medium + Medium Stakes:**
```python
# Add confidence scoring
response = call_model(cot_prompt, temperature=0.0)
confidence = get_confidence_score(response)
if confidence >= 7:
    answer = extract_answer(response)
else:
    # Flag for review
    answer = "UNCERTAIN - Review needed"
```

**For Complex Problem:**
```python
# Use least-to-most
ltm_solver = LeastToMostSolver(call_model)
result = ltm_solver.solve_least_to_most(problem)
answer = extract_answer(result['solution'])
confidence = get_confidence_score(result['solution'])
```

**For High Stakes:**
```python
# Self-consistency with confidence
solver = SelfConsistencySolver(call_model)
result = solver.solve_with_self_consistency(problem, n_samples=10)
answer = result['final_answer']
confidence = result['confidence']  # Already calculated via voting

if confidence >= 0.8:
    # High agreement among 10 paths
    return answer
else:
    # Low agreement - needs investigation
    return "UNCERTAIN - Multiple paths disagree"
```

### Real-World Examples

**Example 1: Customer Support (Medium Stakes)**
```
Problem: "What should we refund to this customer?"
Approach:
  1. Use basic CoT to generate reasoning
  2. Add Confidence Scoring to validate
  3. If confidence < 7: escalate to human
  4. If confidence ≥ 7: process automatically
```

**Example 2: Medical Diagnosis (High Stakes)**
```
Problem: "Based on symptoms, what disease might this be?"
Approach:
  1. Use Self-Consistency with 10 attempts
  2. Check voting confidence
  3. Add Confidence Scoring for each path
  4. Only trust if ≥8/10 paths agree AND ≥8/10 confidence
  5. Otherwise: recommend human doctor review
```

**Example 3: Math Problem Solving (Complex)**
```
Problem: "A store has... [multi-step word problem]"
Approach:
  1. Use Least-to-Most decomposition
  2. Verify with Confidence Scoring
  3. If unsure: use Self-Consistency on final step
  4. Return answer with confidence metric
```

### Measuring Success

After implementing a technique, measure:

```python
# Collect results
accuracies = []
confidences = []
times = []

for problem in test_set:
    start = time.time()
    result = run_technique(problem)
    time_taken = time.time() - start
    
    accuracy = 1 if result['answer'] == true_answer else 0
    confidence = result['confidence']
    
    accuracies.append(accuracy)
    confidences.append(confidence)
    times.append(time_taken)

# Calculate metrics
print(f"Accuracy: {sum(accuracies)/len(accuracies):.1%}")
print(f"Avg Confidence: {sum(confidences)/len(confidences):.1f}/10")
print(f"Avg Time: {sum(times)/len(times):.1f}s per problem")

# Check calibration
high_conf = [a for a, c in zip(accuracies, confidences) if c >= 8]
low_conf = [a for a, c in zip(accuracies, confidences) if c < 5]

print(f"High confidence accuracy: {sum(high_conf)/len(high_conf):.1%}")
print(f"Low confidence accuracy: {sum(low_conf)/len(low_conf):.1%}")
# Good calibration: high confidence ≈ high accuracy
```

---

In [ ]:
class LeastToMostSolver:
    """Break problems into simpler subproblems"""
    
    def __init__(self, model_fn):
        self.model = model_fn
    
    def solve_least_to_most(self, problem: str) -> Dict:
        """Solve using least-to-most prompting"""
        
        # Step 1: Decompose
        decompose_prompt = f"Break this into simple steps: {problem}\n\nSteps:"
        decomposition = self.model(decompose_prompt, max_tokens=150)
        
        # Step 2: Solve each step
        solve_prompt = f"Now solve: {problem}\n\nUsing step-by-step reasoning:"
        solution = self.model(solve_prompt, temperature=0.0, max_tokens=200)
        
        return {
            'problem': problem,
            'decomposition': decomposition,
            'solution': solution,
            'method': 'least-to-most'
        }

ltm_solver = LeastToMostSolver(call_model)

print("\n" + "="*80)
print("LEAST-TO-MOST PROMPTING")
print("="*80)

prob = "Sarah has 10 apples. She gives 3 to John and buys 5 more. How many does she have?"
result = ltm_solver.solve_least_to_most(prob)

print(f"\n📝 Problem: {prob}")
print(f"\n📋 Decomposition:\n{result['decomposition'][:200]}")
print(f"\n✅ Solution:\n{result['solution'][:200]}")

## Confidence Scoring

In [ ]:
def get_confidence_score(problem: str, model_fn) -> Dict:
    """Get model's confidence in its answer"""
    
    # Get reasoning
    reasoning_prompt = f"Let me think step by step. {problem}"
    reasoning = model_fn(reasoning_prompt, max_tokens=200)
    
    # Get confidence
    confidence_prompt = f"Based on this reasoning, how confident are you (1-10)? {reasoning}\n\nConfidence:"
    confidence_response = model_fn(confidence_prompt, max_tokens=50)
    
    # Extract number
    numbers = re.findall(r'\d+', confidence_response)
    confidence = int(numbers[0]) if numbers else 5
    
    return {
        'problem': problem,
        'reasoning': reasoning,
        'confidence': min(confidence, 10),
        'confidence_explanation': confidence_response
    }

print("\n" + "="*80)
print("CONFIDENCE SCORING")
print("="*80)

conf_result = get_confidence_score("What is 2 + 2?", call_model)
print(f"\n📝 Problem: {conf_result['problem']}")
print(f"📊 Confidence: {conf_result['confidence']}/10")

## Compare Techniques

In [ ]:
# Compare methods on same problem
comparison = pd.DataFrame([
    {'Method': 'Self-Consistency (3 samples)', 'Samples': 3, 'Confidence': sc_results[0]['confidence']*100 if sc_results else 0},
    {'Method': 'Least-to-Most', 'Samples': 1, 'Confidence': 75},
    {'Method': 'Single CoT', 'Samples': 1, 'Confidence': 60},
])

print("\n" + "="*80)
print("TECHNIQUE COMPARISON")
print("="*80)
print("\n", comparison.to_string(index=False))

# Visualize
plt.figure(figsize=(10, 5))
plt.barh(comparison['Method'], comparison['Confidence'], color=['#4ECDC4', '#FFE66D', '#FF6B6B'], alpha=0.7, edgecolor='black', linewidth=2)
plt.xlabel('Confidence (%)', fontsize=12, fontweight='bold')
plt.title('Comparison of Advanced CoT Techniques', fontsize=14, fontweight='bold')
plt.xlim(0, 100)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('advanced_techniques_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Comparison plot saved")

## Save Results

In [ ]:
results = {
    'self_consistency': sc_results,
    'least_to_most': result,
    'confidence': conf_result,
}

with open('phase3_results.json', 'w') as f:
    # Convert to serializable format
    json.dump(results, f, indent=2, default=str)

print("✅ Results saved to phase3_results.json")

print("\n" + "="*80)
print("🎉 PHASE 3 COMPLETE")
print("="*80)
print("\nYou've mastered:")
print("✓ Self-Consistency Sampling")
print("✓ Least-to-Most Prompting") 
print("✓ Confidence Scoring")
print("✓ Technique Comparison")
print("\nReady for Phase 4: Real Applications!")